<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week1_Day3_Vector_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install sentence-transformers rank-bm25 numpy

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPOSITORY_NAME"

repo_dir = Path("/content") / REPO_NAME
repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    print(f"Using existing clone: {repo_dir}")

sys.path.insert(0, str(repo_dir))
print("Repository:", repo_dir)

In [ ]:
# Only needed when the repo is already cloned and GitHub has newer commits:
# !git -C /content/YOUR_REPOSITORY_NAME pull

In [ ]:
import search_utils

print("Imported utility from:", search_utils.__file__)
print("Embedding model:", search_utils.MODEL_NAME)

assert Path(search_utils.__file__).resolve() == (
    repo_dir / "search_utils.py"
).resolve(), "Python imported a different search_utils.py file!"

In [ ]:
DATA_FILE = repo_dir / "pubmed_abstracts.json"

# If your JSON is on Google Drive instead, use:
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_FILE = Path(
#     "/content/drive/MyDrive/ontology-aware-litminer/pubmed_abstracts.json"
# )

print("Looking for data at:", DATA_FILE)
print("File exists:", DATA_FILE.exists())

documents = search_utils.load_pubmed_data(DATA_FILE)
print(f"Loaded {len(documents)} documents.")
print("First PMID:", documents[0]["pmid"])
print("First title:", documents[0]["title"])

In [ ]:
import hashlib
import json
import numpy as np

texts = [
    f"{doc['title']}. {doc['abstract']}"
    for doc in documents
]

fingerprint_input = json.dumps(
    {
        "model": search_utils.MODEL_NAME,
        "documents": [
            {
                "pmid": doc["pmid"],
                "title": doc["title"],
                "abstract": doc["abstract"],
            }
            for doc in documents
        ],
    },
    sort_keys=True,
).encode("utf-8")

fingerprint = hashlib.sha256(fingerprint_input).hexdigest()[:16]
cache_file = repo_dir / f"day3_embeddings_{fingerprint}.npy"

if cache_file.exists():
    doc_embeddings = np.load(cache_file)
    print("Loaded cached embeddings:", cache_file)
else:
    doc_embeddings = search_utils.generate_embeddings(texts)
    np.save(cache_file, doc_embeddings)
    print("Saved embeddings:", cache_file)

print("Embeddings shape:", doc_embeddings.shape)
print("Documents:", len(documents))

assert doc_embeddings.ndim == 2
assert doc_embeddings.shape[0] == len(documents)
assert np.allclose(
    np.linalg.norm(doc_embeddings, axis=1),
    1.0,
    atol=1e-3,
)

In [ ]:
query = "human brain cell types identified by single-cell transcriptomics"

vector_results = search_utils.search_vectors(
    query=query,
    doc_embeddings=doc_embeddings,
    documents=documents,
    top_k=5,
)

print("VECTOR SEARCH")
print("Query:", query)

for rank, result in enumerate(vector_results, start=1):
    doc = result["doc"]
    print(
        f"{rank}. cosine={result['score']:.3f} | "
        f"PMID={doc['pmid']} | {doc['title']}"
    )

In [ ]:
bm25_index = search_utils.build_bm25_index(documents)
bm25_results = search_utils.search_bm25(
    bm25_index=bm25_index,
    documents=documents,
    query=query,
    top_k=5,
)

print("BM25 SEARCH")
print("Query:", query)

for rank, result in enumerate(bm25_results, start=1):
    doc = result["doc"]
    print(
        f"{rank}. BM25={result['score']:.3f} | "
        f"PMID={doc['pmid']} | {doc['title']}"
    )